In [20]:
import sys; sys.path.append("..")
import json
from pathlib import Path
import numpy as np
import pandas as pd
from src.data_loader import PROCESSED_DIR
from src.preprocessing import build_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, classification_report
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

In [21]:
X_train = pd.read_parquet(PROCESSED_DIR / "X_train.parquet")
X_test  = pd.read_parquet(PROCESSED_DIR / "X_test.parquet")
y_train = pd.read_parquet(PROCESSED_DIR / "y_train.parquet")["readmitted_30"]
y_test  = pd.read_parquet(PROCESSED_DIR / "y_test.parquet")["readmitted_30"]
groups_train = pd.read_parquet(PROCESSED_DIR / "groups_train.parquet")["patient_nbr"]

print(X_train.shape, X_test.shape)

(69613, 44) (29727, 44)


In [22]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline = Pipeline([
    ("pre",   build_preprocessor()),
    ("scale", StandardScaler()),
    ("clf",   LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
print(baseline)

Pipeline(steps=[('pre',
                 Pipeline(steps=[('prep',
                                  Pipeline(steps=[('labs',
                                                   FunctionTransformer(func=<function encode_labs at 0x11eb729e0>)),
                                                  ('meds',
                                                   FunctionTransformer(func=<function encode_meds at 0x1401901f0>)),
                                                  ('diags',
                                                   FunctionTransformer(func=<function group_diagnoses at 0x140b9c820>))])),
                                 ('onehot',
                                  ColumnTransformer(remainder='passthrough',
                                                    transformers=[('o...
                                                                   OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                                 min_

In [23]:
baseline.fit(X_train, y_train)

,steps,"[('pre', ...), ('scale', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('prep', ...), ('onehot', ...)]"
,transform_input,None
,memory,None
,verbose,False
,steps,"[('labs', ...), ('meds', ...), ...]"
,transform_input,None
,memory,None


In [24]:
proba_train = baseline.predict_proba(X_train)[:, 1]
proba_test  = baseline.predict_proba(X_test)[:, 1]

In [25]:
train_roc, train_pr = roc_auc_score(y_train, proba_train), average_precision_score(y_train, proba_train)
test_roc,  test_pr  = roc_auc_score(y_test,  proba_test),  average_precision_score(y_test,  proba_test)
print("TRAIN  ROC-AUC: %.3f   PR-AUC: %.3f" % (train_roc, train_pr))
print("TEST   ROC-AUC: %.3f   PR-AUC: %.3f" % (test_roc, test_pr))

TRAIN  ROC-AUC: 0.671   PR-AUC: 0.217
TEST   ROC-AUC: 0.670   PR-AUC: 0.220


In [26]:
pred = (proba_test >= 0.5).astype(int)
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, digits=3))

[[18096  8232]
 [ 1522  1877]]
              precision    recall  f1-score   support

           0      0.922     0.687     0.788     26328
           1      0.186     0.552     0.278      3399

    accuracy                          0.672     29727
   macro avg      0.554     0.620     0.533     29727
weighted avg      0.838     0.672     0.729     29727



In [27]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    baseline, X_train, y_train,
    groups=groups_train, cv=cv,
    scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
    n_jobs=-1,
)
cv_roc, cv_pr = scores["test_roc_auc"], scores["test_pr_auc"]
print("CV ROC-AUC: %.3f ± %.3f" % (cv_roc.mean(), cv_roc.std()))
print("CV PR-AUC : %.3f ± %.3f" % (cv_pr.mean(), cv_pr.std()))

CV ROC-AUC: 0.661 ± 0.009
CV PR-AUC : 0.212 ± 0.007


In [28]:
Path("../reports").mkdir(exist_ok=True)
results = {
    "model": "preprocessor + StandardScaler + LogisticRegression(balanced)",
    "n_features": int(baseline.named_steps["pre"].transform(X_train.head()).shape[1]),
    "cv_roc_auc_mean": float(cv_roc.mean()), "cv_roc_auc_std": float(cv_roc.std()),
    "cv_pr_auc_mean": float(cv_pr.mean()),  "cv_pr_auc_std": float(cv_pr.std()),
    "test_roc_auc": float(test_roc), "test_pr_auc": float(test_pr),
}
json.dump(results, open("../reports/baseline_results.json", "w"), indent=2)
results

{'model': 'preprocessor + StandardScaler + LogisticRegression(balanced)',
 'n_features': 138,
 'cv_roc_auc_mean': 0.660982331645343,
 'cv_roc_auc_std': 0.008896238893463658,
 'cv_pr_auc_mean': 0.21226089102806842,
 'cv_pr_auc_std': 0.006679641156286915,
 'test_roc_auc': 0.6700253077276469,
 'test_pr_auc': 0.21972898958801637}